# Neural Decoding of Imagined Handwriting — Benchmark

**Comparison: Alignment Methods × Decoder Architectures**

| | GRU (Willett) | RCNN | Conformer | CTC |
|---|---|---|---|---|
| **Gaussian HMM (Willett)** | baseline | new | **new** | N/A |
| **Poisson HMM (ours)** | new | new | **new** | N/A |
| **No alignment** | N/A | N/A | N/A | new |

This notebook runs the full benchmark on GPU/TPU.

**Runtime → Change runtime type → GPU (or TPU if available)**

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/YOUR_USERNAME/neural_decoding_imagined_handwriting.git
%cd neural_decoding_imagined_handwriting

In [ ]:
# Install dependencies
!pip install -q torch scipy scikit-learn h5py numpy

In [ ]:
# Verify GPU/TPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Download Dataset

The Willett dataset is hosted on Dryad. Dryad uses bot protection,
so we download via a workaround.

**Option A:** If you have the tarball locally, upload it to Colab:
- Click the folder icon (left sidebar) → Upload → select `handwritingBCIData.tar.gz`

**Option B:** Upload to Google Drive first, then mount:

In [ ]:
import os

# === OPTION A: Direct upload ===
# If you uploaded the tarball to Colab's file browser:
TARBALL = '/content/handwritingBCIData.tar.gz'

# === OPTION B: Google Drive ===
# Uncomment these lines if you put the tarball in your Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# TARBALL = '/content/drive/MyDrive/handwritingBCIData.tar.gz'

if os.path.exists(TARBALL):
    print(f'Found tarball: {TARBALL}')
    if not os.path.exists('handwritingBCIData/Datasets'):
        !tar -xzf {TARBALL}
        print('Extracted.')
    else:
        print('Already extracted.')
else:
    print(f'Tarball not found at {TARBALL}')
    print('Upload handwritingBCIData.tar.gz to Colab or Google Drive.')
    print('Download from: https://datadryad.org/dataset/doi:10.5061/dryad.wh70rxwmv')

In [ ]:
# Verify dataset
!ls handwritingBCIData/Datasets/

## 3. Quick Sanity Check

Load one session and verify the data pipeline works.

In [ ]:
from data import WillettDataset
import numpy as np

ds = WillettDataset('./handwritingBCIData')
sessions = ds.list_sessions()
print(f'{len(sessions)} sessions: {sessions}')

session = ds.load_session(sessions[0])
print(f'\nSession {session["session_id"]}:')
print(f'  Neural: {session["neural"].shape}')
print(f'  Labels: {len(np.unique(session["char_labels"]))} classes')

X_tr, y_tr, X_te, y_te = ds.get_train_test_split(sessions[0])
print(f'  Train: {X_tr.shape}, Test: {X_te.shape}')

## 4. Run Benchmark (Fast)

Quick run with small models to verify everything works.
Takes ~5 minutes on GPU.

In [ ]:
!python3 run_benchmark.py --max-len 1500

## 5. Run Benchmark (Full)

Larger models, more epochs, full-length sentences.
Takes ~30-60 minutes on GPU.

In [ ]:
!python3 run_benchmark.py --full --max-len 3000

## 6. Multi-Session Benchmark

Run across all 10 sessions for statistically meaningful results.
This takes longer but produces confidence intervals.

In [ ]:
from run_benchmark import main as run_benchmark
from data import WillettDataset
import sys

ds = WillettDataset('./handwritingBCIData')
sessions = ds.list_sessions()

all_results = {}
for session_id in sessions:
    print(f'\n{"#"*70}')
    print(f'# SESSION: {session_id}')
    print(f'{"#"*70}')
    sys.argv = [
        'run_benchmark.py',
        '--session', session_id,
        '--full',
        '--max-len', '3000',
    ]
    try:
        results = run_benchmark()
        all_results[session_id] = results
    except Exception as e:
        print(f'  FAILED: {e}')
        continue

In [ ]:
# Aggregate results across sessions
import numpy as np
from collections import defaultdict

agg = defaultdict(lambda: defaultdict(list))
for session_id, results in all_results.items():
    for (align, dec), metrics in results.items():
        key = f'{dec} + {align}'
        for metric, val in metrics.items():
            agg[key][metric].append(val)

print('=' * 80)
print('AGGREGATE RESULTS (mean ± std across sessions)')
print('=' * 80)
print(f'{"Configuration":<45s} {"CER":>10s} {"WER":>10s} {"Acc":>10s}')
print('-' * 80)
for key in sorted(agg.keys()):
    m = agg[key]
    cer = np.array(m['cer']) * 100
    wer = np.array(m['wer']) * 100
    acc = np.array(m['frame_acc']) * 100
    print(f'{key:<45s} '
          f'{cer.mean():>5.1f}±{cer.std():>4.1f}% '
          f'{wer.mean():>5.1f}±{wer.std():>4.1f}% '
          f'{acc.mean():>5.1f}±{acc.std():>4.1f}%')

## 7. Visualise Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Parse results into plottable form
configs = sorted(agg.keys())
cer_means = [np.mean(agg[k]['cer']) * 100 for k in configs]
cer_stds  = [np.std(agg[k]['cer']) * 100 for k in configs]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(configs)), cer_means, xerr=cer_stds,
               color=['#2196F3' if 'Poisson' in c else
                      '#FF9800' if 'Gaussian' in c else
                      '#4CAF50' for c in configs],
               edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(configs)))
ax.set_yticklabels(configs, fontsize=10)
ax.set_xlabel('Character Error Rate (%)', fontsize=12)
ax.set_title('Neural Handwriting Decoding: Alignment × Decoder Comparison', fontsize=14)
ax.invert_yaxis()
ax.axvline(x=2.78, color='red', linestyle='--', label='Willett (2021) raw CER')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150)
plt.show()
print('Saved to benchmark_results.png')

## Notes

- **Willett's reported CER**: 2.78% (HeldOutTrials, raw, no LM) with 10 sessions,
  synthetic data augmentation, and thousands of training epochs.
- Our benchmark uses 1 session, no augmentation, and fewer epochs.
  Results will be worse, but the **relative comparison** between methods is valid.
- To match Willett's setup: train on all 10 sessions (Section 6),
  use `--max-len 0` (no truncation), and increase epochs.
- The CTC decoder is alignment-free — it appears once in the table
  regardless of alignment method.
- Blue bars = Poisson HMM (ours), Orange = Gaussian HMM (Willett),
  Green = CTC (no alignment).